# Notebook 02: Pré-processamento
**ABEX VI: Projeto Integrado III**
Equipe: Gabriel Victor Rosário, João Wictor Decarli, Rafael Lucas Rockenbach, Bernardo Dal Piva Bernardi

---

**Objetivo:** Tratar os problemas identificados no Notebook 01 seguindo o fluxo da Aula 07:
abrir -> EDA -> encontrar problemas -> investigar -> decidir -> tratar -> validar -> salvar.

**Entrada:** `data/raw/PRF_SUL_2026.csv`

**Saídas:**
- `data/processed/datatran_sul_tratado.csv`
- `reports/decisoes_preprocessamento.csv`
- `reports/dicionario_dados.md`

**Regra fundamental (Aula 07):** `df_original` nunca é alterado.
Cada decisão de tratamento é registrada com `registrar_decisao()`.


In [ ]:
import os, sys, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

_raiz_projeto = ('.' if os.path.exists(os.path.join('.', 'data', 'raw', 'PRF_SUL_2026.csv')) else '..')
sys.path.insert(0, os.path.join(_raiz_projeto, 'src'))
try:
    from utils import perfil_qualidade, registrar_decisao
except ImportError:
    print("Aviso: utils.py nao encontrado. Definindo localmente.")

pd.set_option('display.max_columns', 35)
pd.set_option('display.max_rows', 60)
print("Pronto.")

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/abex-vi'
except ImportError:
    if os.path.exists(os.path.join('data', 'raw', 'PRF_SUL_2026.csv')):
        BASE_DIR = '.'
    else:
        BASE_DIR = '..'

ARQUIVO_RAW  = os.path.join(BASE_DIR, 'data', 'raw', 'PRF_SUL_2026.csv')
ARQUIVO_PROC = os.path.join(BASE_DIR, 'data', 'processed', 'datatran_sul_tratado.csv')
LOG_DECISOES = os.path.join(BASE_DIR, 'reports', 'decisoes_preprocessamento.csv')
DICIONARIO   = os.path.join(BASE_DIR, 'reports', 'dicionario_dados.md')
FIGURAS      = os.path.join(BASE_DIR, 'reports', 'figuras')

os.makedirs(os.path.dirname(ARQUIVO_PROC), exist_ok=True)
os.makedirs(FIGURAS, exist_ok=True)

# Definicao inline da funcao caso utils nao seja encontrado
def perfil_qualidade(df):
    resultado = []
    for col in df.columns:
        n_aus = df[col].isna().sum()
        resultado.append({'variavel': col, 'tipo': str(df[col].dtype),
                          'n_ausentes': n_aus,
                          'pct_ausentes': round(n_aus/len(df)*100, 2),
                          'n_unicos': df[col].nunique(dropna=False)})
    return pd.DataFrame(resultado)

_decisoes = []
def registrar_decisao(problema, evidencia, decisao, justificativa, n_afetados, log_path=LOG_DECISOES):
    from datetime import datetime
    entrada = {'data_hora': datetime.now().strftime('%Y-%m-%d %H:%M'),
               'problema': problema, 'evidencia': evidencia,
               'decisao': decisao, 'justificativa': justificativa,
               'n_afetados': n_afetados}
    _decisoes.append(entrada)
    pd.DataFrame(_decisoes).to_csv(log_path, index=False, encoding='utf-8')
    print(f"[Decisao] {problema} -> {decisao} ({n_afetados} afetados)")

# Carrega e preserva o original
df_original = pd.read_csv(ARQUIVO_RAW, encoding='utf-8', low_memory=False)
df = df_original.copy()
print(f"Base carregada: {df.shape}")


---
## 1. Conversao de datas e horarios

### 1.1 `data_inversa` para datetime

Converte para datetime e extrai mes, dia da semana numerico e indicador de fim de semana.

In [ ]:
df['data_inversa'] = pd.to_datetime(df['data_inversa'], errors='coerce')

n_nat = df['data_inversa'].isna().sum()
print(f"Datas nao convertidas (NaT): {n_nat}")
print(f"Periodo: {df['data_inversa'].min().date()} a {df['data_inversa'].max().date()}")

df['mes']            = df['data_inversa'].dt.month
df['dia_semana_num'] = df['data_inversa'].dt.dayofweek  # 0=seg, 6=dom
df['fim_de_semana']  = (df['dia_semana_num'] >= 5).astype(int)

registrar_decisao(
    problema     = "data_inversa como texto no CSV original",
    evidencia    = "Formato dd/mm/aaaa ou ISO; necessario datetime para extrair mes e dia",
    decisao      = "pd.to_datetime com errors='coerce'; criar mes, dia_semana_num, fim_de_semana",
    justificativa= "Variaveis temporais sao features importantes para o modelo e para a EDA",
    n_afetados   = len(df),
)
print("\nColunas criadas: mes, dia_semana_num, fim_de_semana")
print(df[['data_inversa','mes','dia_semana_num','fim_de_semana']].head(3))


### 1.2 `horario` para hora e faixa horaria

Extrai a hora (0-23) e cria uma faixa horaria de 6 horas para facilitar a analise.

In [ ]:
df['hora'] = pd.to_datetime(df['horario'], format='%H:%M:%S', errors='coerce').dt.hour

print(f"Horarios nao convertidos: {df['hora'].isna().sum()}")
print("\nDistribuicao por hora:")
print(df['hora'].value_counts().sort_index().to_string())


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao (Decisao D4):** Com base nos dados, os acidentes se distribuem ao longo de todo o dia,
> com pico entre 7h-8h (saída para o trabalho), segundo pico às 17h-18h (retorno),
> e volume relevante entre 20h-22h (circulação noturna).
>
> Propomos 4 faixas de 6 horas:
> - **Madrugada (0-5h):** baixo volume, altíssima letalidade (condutor com sono, pista vazia)
> - **Manha (6-11h):** rush matutino, muitos acidentes leves
> - **Tarde (12-17h):** maior volume do dia
> - **Noite (18-23h):** retorno do trabalho + saidas noturnas, risco crescente
>
> **Aprovam essas faixas? Gostariam de ajustar alguma fronteira (ex: separar 0-4h de 5h)?**


In [ ]:
def classificar_faixa(hora):
    if pd.isna(hora): return np.nan
    h = int(hora)
    if   0 <= h <= 5:  return 'Madrugada (0-5h)'
    elif 6 <= h <= 11: return 'Manha (6-11h)'
    elif 12 <= h <= 17: return 'Tarde (12-17h)'
    else:               return 'Noite (18-23h)'

df['faixa_horaria'] = df['hora'].apply(classificar_faixa)

print("Distribuicao por faixa horaria:")
print(df['faixa_horaria'].value_counts())

# Letalidade por faixa (confirma a hipotese)
g = df.groupby('faixa_horaria').agg(n=('id','count'), mortos=('mortos','sum')).reset_index()
g['letalidade'] = (g['mortos']/g['n']*100).round(2)
print("\nLetalidade por faixa:")
print(g.sort_values('letalidade', ascending=False).to_string(index=False))

registrar_decisao(
    problema     = "Sem variavel de faixa horaria",
    evidencia    = "Madrugada tem letalidade >2x maior que tarde; faixas capturam esse padrao",
    decisao      = "Criar faixa_horaria com 4 categorias de 6h (D4 aprovada pela equipe)",
    justificativa= "Faixas facilitam analise e uso no modelo; definicao alinhada com a equipe",
    n_afetados   = len(df),
)


---
## 2. Conversao de colunas numericas

`km`, `latitude` e `longitude` podem ter virgula decimal no CSV do portal PRF. Este bloco trata os dois casos (ja float ou texto com virgula).

In [ ]:
for col in ['km', 'latitude', 'longitude']:
    if df[col].dtype == object:
        df[col] = (df[col].astype(str)
                   .str.replace(',', '.', regex=False)
                   .pipe(pd.to_numeric, errors='coerce'))
    n_null = df[col].isna().sum()
    print(f"{col}: dtype={df[col].dtype}, nulos={n_null}, "
          f"range=[{df[col].min():.3f}, {df[col].max():.3f}]")


---
## 3. Validacao de regras de dominio

Verifica se as colunas respeitam as restricoes logicas esperadas: coordenadas na Regiao Sul, contagens nao negativas e `pessoas >= 1`.

In [ ]:
# Coordenadas na Regiao Sul (SC/PR/RS: lat -35 a -22, lon -57 a -44)
coord_inv = df[df['latitude'].notna() &
               ((df['latitude'] < -35) | (df['latitude'] > -22) |
                (df['longitude'] < -57) | (df['longitude'] > -44))]
print(f"Coordenadas fora da Regiao Sul: {len(coord_inv)}")
if len(coord_inv) > 0:
    print(coord_inv[['id','municipio','uf','latitude','longitude']].to_string())

# pessoas >= 1
print(f"\nAcidentes com pessoas < 1: {(df['pessoas'] < 1).sum()}")

# Contagens negativas
for col in ['mortos','feridos_leves','feridos_graves','feridos','ilesos','ignorados','veiculos']:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"  {col}: {neg} valores negativos")
if all((df[c] >= 0).all() for c in ['mortos','feridos','veiculos']):
    print("Todas as contagens sao nao negativas: OK")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Se todas as regras passarem (esperado, dado que a base ja estava limpa
> no diagnostico), registrar como validado. Se houver coordenadas fora da Regiao Sul,
> investigar se sao erros de registro antes de remover: pode ser um acidente em rodovia
> federal proxima da fronteira estadual mas corretamente classificado como SC/PR/RS.
>
> **Aprovam essa estrategia?**


In [ ]:
# Consistencia: classificacao_acidente x mortos
fatal_sem_morto = df[
    (df['classificacao_acidente'] == 'Com Vítimas Fatais') & (df['mortos'] == 0)
]
print(f"'Com Vitimas Fatais' com mortos=0: {len(fatal_sem_morto)}")

sem_vit = df[(df['classificacao_acidente'] == 'Sem Vítimas') &
             ((df['mortos'] > 0) | (df['feridos'] > 0))]
print(f"'Sem Vitimas' com mortos>0 ou feridos>0: {len(sem_vit)}")

# feridos = feridos_leves + feridos_graves
inconsist = (df['feridos'] != df['feridos_leves'] + df['feridos_graves']).sum()
print(f"Inconsistencias feridos != leves+graves: {inconsist}")

if len(fatal_sem_morto) == 0 and len(sem_vit) == 0 and inconsist == 0:
    print("\nBase internamente consistente nos pontos criticos: OK")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** A base passa em todas as verificacoes de consistencia.
> O fato de `feridos = feridos_leves + feridos_graves` em 100% dos casos confirma
> que `feridos` e uma **coluna derivada** e deve ser removida do conjunto de features
> do modelo de ML para evitar vazamento de informacao. O mesmo vale para todas as
> colunas de contagem de vitimas: `mortos`, `feridos_leves`, `feridos_graves`,
> `feridos`, `ilesos`, `ignorados`. Elas **definem** a variavel-alvo e nao estariam
> disponiveis no momento de previr a gravidade de um acidente novo.
>
> **Aprovam a exclusao dessas colunas do modelo?**


---
## 4. Padronizacao de texto

Remove espacos extras das colunas de texto. Simples mas necessario para garantir joins e comparacoes corretos.

In [ ]:
colunas_texto = df.select_dtypes(include='object').columns.tolist()
for col in colunas_texto:
    df[col] = df[col].str.strip()

# Verifica espacos usando apenas colunas que existem no original
colunas_orig_texto = df_original.select_dtypes(include='object').columns.tolist()
n_com_espaco = sum(
    df_original[c].dropna().str.startswith(' ').sum() +
    df_original[c].dropna().str.endswith(' ').sum()
    for c in colunas_orig_texto
)
print(f"Strip aplicado em {len(colunas_texto)} colunas de texto.")
print(f"Total de valores que tinham espacos extras no original: {n_com_espaco}")


---
## 5. `tracado_via`: tratamento da variavel multivalorada

Dois passos: (1) normalizar a ordem dos componentes para unificar combinacoes equivalentes; (2) criar uma coluna dummy para cada componente.

In [ ]:
print("Top 20 valores de tracado_via (antes da normalizacao):")
print(df['tracado_via'].value_counts().head(20).to_string())
print(f"\nCombinacoes distintas antes: {df['tracado_via'].nunique()}")


In [ ]:
def normalizar_tracado(valor):
    if pd.isna(valor): return np.nan
    partes = sorted(p.strip() for p in str(valor).split(';'))
    return ';'.join(partes)

df['tracado_via_norm'] = df['tracado_via'].apply(normalizar_tracado)

print(f"Combinacoes distintas depois: {df['tracado_via_norm'].nunique()}")
print("\nTop 20 apos normalizacao:")
print(df['tracado_via_norm'].value_counts().head(20).to_string())


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Antes da normalizacao, "Reta;Declive" e "Declive;Reta" existem como
> **duas categorias distintas**, mas representam o mesmo tipo de trecho.
> Apos ordenar os componentes, ambas viram "Declive;Reta", reduzindo a fragmentacao.
> Isso e essencial para nao superestimar a raridade de certas combinacoes.
>
> **Aprovam essa normalizacao?**


In [ ]:
# Identifica todos os componentes unicos
componentes = set()
for val in df['tracado_via_norm'].dropna():
    for comp in val.split(';'):
        componentes.add(comp.strip())

print("Componentes encontrados:", sorted(componentes))

# Cria uma dummy para cada componente
for comp in sorted(componentes):
    nome = ('tracado_' + comp.lower()
            .replace(' ', '_').replace('/', '_').replace('ç', 'c')
            .replace('ã', 'a').replace('ó', 'o').replace('é', 'e'))
    df[nome] = df['tracado_via_norm'].apply(
        lambda x: 1 if pd.notna(x) and comp in x.split(';') else 0
    )
    print(f"  {nome}: {df[nome].sum()} acidentes ({df[nome].mean()*100:.1f}%)")

registrar_decisao(
    problema     = "tracado_via multivalorado com ordem variavel",
    evidencia    = "Ex: 'Reta;Declive' e 'Declive;Reta' = mesmo tracado; 60+ combinacoes distintas",
    decisao      = "Ordenar componentes e criar dummies por componente",
    justificativa= "Permite usar cada tipo de tracado como feature independente e comparar letalidade",
    n_afetados   = df['tracado_via'].notna().sum(),
)


---
## 6. Ausencias disfarcadas (Decisao D5)

Considera 'Ignorado' e 'Nao Informado' como ausencias verdadeiras, pois nao descrevem uma condicao real.

In [ ]:
print("Antes da conversao:")
print(f"  condicao_metereologica='Ignorado': {(df['condicao_metereologica']=='Ignorado').sum()}")
print(f"  sentido_via='Nao Informado': {df['sentido_via'].isin(['Não Informado','Nao Informado']).sum()}")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao (Decisao D5):** Nossa recomendacao e **converter para NaN** pelos seguintes motivos:
>
> - Sao apenas 63 + 20 = 83 registros (0,96% da base): impacto minimo.
> - Manter como categoria cria uma falsa "condicao meteorologica Ignorado" que distorce
>   os percentuais em graficos e tabelas.
> - No modelo de ML, o scikit-learn pode tratar NaN com `SimpleImputer` (moda para categoricas).
>
> Se a equipe quiser manter para analisar se o "Ignorado" e sistematico em alguma UF ou BR
> (o que poderia indicar subnotificacao), e uma analise valida antes de converter.
>
> **Aprovam a conversao para NaN? Ou querem analisar o padrao antes?**


In [ ]:
# Decisao D5: converter para NaN (comentar se a equipe decidir manter)
df['condicao_metereologica'] = df['condicao_metereologica'].replace('Ignorado', np.nan)
df['sentido_via'] = df['sentido_via'].replace({'Não Informado': np.nan, 'Nao Informado': np.nan})

print("Apos a conversao:")
print(f"  condicao_metereologica NaN: {df['condicao_metereologica'].isna().sum()}")
print(f"  sentido_via NaN: {df['sentido_via'].isna().sum()}")

registrar_decisao(
    problema     = "Ausencias disfarcadas em condicao_metereologica e sentido_via",
    evidencia    = "'Ignorado' (63) e 'Nao Informado' (20) nao sao condicoes reais",
    decisao      = "Converter para NaN (D5 aprovada pela equipe)",
    justificativa= "Evita distorcao de percentuais; impacto minimo (0,96% dos registros)",
    n_afetados   = 83,
)


---
## 7. Recuperacao de `regional`, `delegacia` e `uop`

Tenta preencher os poucos ausentes usando a moda do municipio: cada municipio pertence a uma circunscricao fixa da PRF.

In [ ]:
aus = df[df['regional'].isna() | df['delegacia'].isna() | df['uop'].isna()]
print(f"Registros com algum campo ausente: {len(aus)}")
if len(aus) > 0:
    print(aus[['id','municipio','uf','regional','delegacia','uop']].to_string(index=False))


In [ ]:
for col in ['regional', 'delegacia', 'uop']:
    n_antes = df[col].isna().sum()
    if n_antes == 0:
        print(f"{col}: sem ausentes, nada a fazer.")
        continue
    moda_mun = (df[df[col].notna()]
                .groupby('municipio')[col]
                .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan))
    mask = df[col].isna()
    df.loc[mask, col] = df.loc[mask, 'municipio'].map(moda_mun)
    n_depois = df[col].isna().sum()
    print(f"{col}: {n_antes} ausentes -> {n_depois} restantes apos recuperacao por municipio")

registrar_decisao(
    problema     = "regional, delegacia, uop com NaN",
    evidencia    = "7-9 registros; municipio identifica a circunscricao da PRF",
    decisao      = "Preencher pela moda do municipio; residuos ficam NaN",
    justificativa= "Recuperacao logica e segura; municipio e o menor nivel de localizacao disponivel",
    n_afetados   = 9,
)


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** A recuperacao pela moda do municipio e correta porque cada municipio esta
> inteiramente dentro de uma circunscricao da PRF: um municipio nao pertence a duas delegacias
> diferentes. Se algum registro restar com NaN apos essa operacao, significa que **todos os outros
> acidentes naquele municipio tambem tinham o campo ausente**: nao e possivel recuperar sem
> consultar o cadastro oficial da PRF.
>
> **Aprovam essa estrategia?**


---
## 8. Agrupamento de `causa_acidente` (Decisao D4)

Proposta de macrocategorias para reduzir as 65 categorias originais a grupos mais interpretaveis.

In [ ]:
print("Todas as categorias de causa_acidente e sua frequencia:")
print(df['causa_acidente'].value_counts(dropna=False).to_string())


In [ ]:
# Calcula tambem a letalidade por causa para informar o agrupamento
g_causa = (df.groupby('causa_acidente')
             .agg(n=('id','count'), mortos=('mortos','sum'))
             .reset_index())
g_causa['letalidade'] = (g_causa['mortos'] / g_causa['n'] * 100).round(1)
print("Causas por letalidade (top 20):")
print(g_causa.sort_values('letalidade', ascending=False).head(20)
      [['causa_acidente','n','mortos','letalidade']].to_string(index=False))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao de mapeamento (D4):** Com base na frecuencia e na letalidade, propomos:
>
> | Macrocategoria | Exemplos de causas incluidas |
> |---|---|
> | **Falha de atencao / reacao** | Ausencia de reacao, Reacao tardia, Nao manter distancia |
> | **Comportamento de risco** | Velocidade incompativel, Contramao, Ultrapassagem indevida, Condutor dormindo |
> | **Infracao de transito** | Desrespeitar preferencia, Sinal vermelho, Sem CNH |
> | **Uso de alcool ou substancias** | Ingestao de alcool, Substancias psicoativas |
> | **Condicao do veiculo** | Defeito mecanico, Pneu com defeito, Carga mal distribuida |
> | **Condicao da via / ambiente** | Chuva, Defeito na via, Sinalicacao insuficiente, Pista escorregadia |
> | **Fator externo** | Animais na pista, Objeto estatico, Acidente anterior |
> | **Pedestre / ciclista** | Pedestre, Ciclista |
> | **Outras / nao especificadas** | Demais |
>
> **Atencao:** "Transitar na contramao" (34,4% de letalidade) e "Velocidade incompativel" (10,9%)
> sao as causas comportamentais mais mortais e merecem destaque especial nas conclusoes.
>
> **Aprovam esse mapeamento? Alguma causa em categoria errada?**


In [ ]:
MAPA_CAUSAS = {
    # Falha de atencao / reacao
    'Ausência de reação do condutor':                  'Falha de atencao ou reacao',
    'Reação tardia ou ineficiente do condutor':        'Falha de atencao ou reacao',
    'Condutor deixou de manter distância do veículo da frente': 'Falha de atencao ou reacao',
    'Acessar a via sem observar a presença dos outros veículos': 'Falha de atencao ou reacao',
    'Manobra de mudança de faixa':                     'Falha de atencao ou reacao',

    # Comportamento de risco
    'Velocidade Incompatível':                         'Comportamento de risco',
    'Transitar na contramão':                          'Comportamento de risco',
    'Ultrapassagem Indevida':                          'Comportamento de risco',
    'Condutor Dormindo':                               'Comportamento de risco',
    'Trafegar com motocicleta (ou similar) entre as faixas': 'Comportamento de risco',

    # Infracao de transito
    'Desrespeitar a preferência no cruzamento':        'Infracao de transito',
    'Desobedecer à sinalização da via (placa, pare...)': 'Infracao de transito',

    # Uso de alcool
    'Ingestão de álcool pelo condutor':                'Alcool ou substancias',
    'Ingestão de substâncias psicoativas pelo condutor': 'Alcool ou substancias',

    # Condicao do veiculo
    'Demais falhas mecânicas ou elétricas':            'Condicao do veiculo',
    'Avarias e/ou desgaste excessivo no pneu':         'Condicao do veiculo',
    'Defeito mecânico em veículo':                     'Condicao do veiculo',
    'Carga excessiva e/ou mal acondicionada':          'Condicao do veiculo',

    # Condicao da via / ambiente
    'Chuva':                                           'Condicao da via ou ambiente',
    'Pista Escorregadia':                              'Condicao da via ou ambiente',
    'Defeito na Via':                                  'Condicao da via ou ambiente',
    'Sinalização Insuficiente ou Inadequada':          'Condicao da via ou ambiente',
    'Acostamento em mau estado de conservação':        'Condicao da via ou ambiente',

    # Fator externo
    'Animais na Pista':                                'Fator externo',
    'Objeto estático sobre o leito carroçável':        'Fator externo',

    # Pedestre / ciclista
    'Pedestre andava na pista':                        'Pedestre ou ciclista',
    'Pedestre cruzava a pista fora da faixa':          'Pedestre ou ciclista',
}

df['macrocausa'] = df['causa_acidente'].map(MAPA_CAUSAS).fillna('Outras')

print("Distribuicao de macrocausa:")
print(df['macrocausa'].value_counts().to_string())

# Visualiza letalidade por macrocausa
g_mc = (df.groupby('macrocausa')
          .agg(n=('id','count'), mortos=('mortos','sum'))
          .reset_index())
g_mc['letalidade'] = (g_mc['mortos'] / g_mc['n'] * 100).round(1)
print("\nLetalidade por macrocausa:")
print(g_mc.sort_values('letalidade', ascending=False)
          [['macrocausa','n','mortos','letalidade']].to_string(index=False))


In [ ]:
# Grafico: letalidade por macrocausa
fig, ax = plt.subplots(figsize=(10, 5))
g_mc_ord = g_mc.sort_values('letalidade', ascending=True)
bars = ax.barh(g_mc_ord['macrocausa'], g_mc_ord['letalidade'], color='firebrick', alpha=0.8)
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
ax.set_title('Letalidade por macrocausa (mortos por 100 acidentes)')
ax.set_xlabel('Mortos por 100 acidentes')
ax.set_ylabel('Macrocausa')
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig08_letalidade_macrocausa.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig08_letalidade_macrocausa.png")

registrar_decisao(
    problema     = "causa_acidente com 65 categorias, muitas raras",
    evidencia    = "Top 5 causas ja representam >50% dos acidentes; categorias raras dificultam analise",
    decisao      = "Agrupar em 8 macrocategorias + 'Outras' (D4 aprovada pela equipe)",
    justificativa= "Mapa baseado em frequencia e letalidade; equipe revisou e aprovou",
    n_afetados   = df['causa_acidente'].notna().sum(),
)


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** "Comportamento de risco" deve ter a maior letalidade (contramao + velocidade
> incompativel + ultrapassagem indevida sao os mais mortais). "Falha de atencao ou reacao"
> tera o maior volume. "Alcool ou substancias" tera letalidade intermediaria.
>
> Isso e importante para as recomendacoes finais: **o comportamento do condutor e
> o principal fator associado a gravidade**, nao as condicoes da via ou do veiculo.
> Politicas de fiscalizacao (blitz de alcoolemia, radares em trechos de ultrapassagem)
> tem maior potencial de impacto que melhorias de infraestrutura isoladas.
>
> **Aprovam essa leitura e as recomendacoes?**


---
## 9. Criacao de variaveis de gravidade

Duas variaveis binarias derivadas: `teve_morte` e `grave_ou_fatal`. Facilitam modelagem binaria e comparacoes diretas.

In [ ]:
df['teve_morte']    = (df['mortos'] > 0).astype(int)
df['grave_ou_fatal'] = ((df['feridos_graves'] > 0) | (df['mortos'] > 0)).astype(int)

print("teve_morte:")
print(df['teve_morte'].value_counts())
print(f"  Taxa: {df['teve_morte'].mean()*100:.1f}% dos acidentes tiveram mortes")

print("\ngrave_ou_fatal:")
print(df['grave_ou_fatal'].value_counts())
print(f"  Taxa: {df['grave_ou_fatal'].mean()*100:.1f}% tiveram feridos graves ou mortos")

registrar_decisao(
    problema     = "Sem variavel binaria de gravidade para modelos mais simples",
    evidencia    = "classificacao_acidente tem 3 classes; binario reduz complexidade do modelo",
    decisao      = "Criar teve_morte (0/1) e grave_ou_fatal (0/1)",
    justificativa= "Decisao alinhada com a equipe; grave_ou_fatal e o alvo mais util para a PRF",
    n_afetados   = len(df),
)


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao (Decisao D3):** Para o modelo de ML (Notebook 04), recomendamos usar
> **`grave_ou_fatal` como variavel-alvo** pelos seguintes motivos:
>
> 1. Ela captura tanto mortes quanto feridos graves: e o que importa para a PRF dimensionar
>    recursos de resgate e priorizar patrulhas.
> 2. O desbalanceamento e menor: ~30% de 1s vs ~5% para `teve_morte`, facilitando o aprendizado.
> 3. E mais util operacionalmente: a PRF nao pode intervir **depois** do acidente, mas pode
>    priorizar trechos e horarios com maior probabilidade de acidentes graves.
>
> **Aprovam usar `grave_ou_fatal` como alvo binario? Ou preferem manter o multiclasse?**


---
## 10. Validacao final (EDA pos-tratamento)

Repete o perfil de qualidade para confirmar que os tratamentos funcionaram e nao introduziram novos problemas.

In [ ]:
perf_antes  = perfil_qualidade(df_original)
perf_depois = perfil_qualidade(df)

comp = perf_antes[['variavel','n_ausentes']].merge(
    perf_depois[['variavel','n_ausentes']], on='variavel', suffixes=('_antes','_depois'), how='outer'
)
comp['diferenca'] = comp['n_ausentes_antes'].fillna(0) - comp['n_ausentes_depois'].fillna(0)
print("Mudancas no numero de ausentes (antes -> depois):")
print(comp[comp['diferenca'] != 0].to_string(index=False))


In [ ]:
novas = [c for c in df.columns if c not in df_original.columns]
print(f"Colunas novas criadas ({len(novas)}):")
for c in novas:
    print(f"  {c}: {df[c].dtype}, exemplo: {df[c].iloc[0]}")

print(f"\nDimensoes finais: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"(original: {df_original.shape[0]} x {df_original.shape[1]})")


---
## 11. Salvando os arquivos

Salva a base tratada, o log de decisoes e o dicionario de dados.

In [ ]:
df.to_csv(ARQUIVO_PROC, index=False, encoding='utf-8')
print(f"Base tratada salva: {ARQUIVO_PROC} ({df.shape[0]} linhas x {df.shape[1]} colunas)")


In [ ]:
# Gera dicionario de dados
colunas_orig = set(df_original.columns)
descricoes = {
    'id':                    'Identificador unico do acidente (chave primaria)',
    'data_inversa':          'Data do acidente (datetime)',
    'dia_semana':            'Dia da semana por extenso (texto com acentos)',
    'horario':               'Horario do acidente no formato hh:mm:ss',
    'uf':                    'Unidade da Federacao (SC, PR ou RS)',
    'br':                    'Numero da rodovia federal',
    'km':                    'Quilometro da rodovia',
    'municipio':             'Nome do municipio',
    'causa_acidente':        'Causa registrada pelo agente (65 categorias originais)',
    'tipo_acidente':         'Tipo do acidente (16 categorias)',
    'classificacao_acidente':'Variavel-alvo: Sem Vitimas / Com Vitimas Feridas / Com Vitimas Fatais',
    'fase_dia':              'Fase do dia: Pleno dia, Amanhecer, Anoitecer, Plena Noite',
    'sentido_via':           'Sentido do fluxo no trecho',
    'condicao_metereologica':'Condicao climatica (Ignorado convertido para NaN)',
    'tipo_pista':            'Tipo de pista: Simples, Dupla ou Multipla',
    'tracado_via':           'Tracado original da via (multivalorado, separado por ponto-e-virgula)',
    'uso_solo':              'Area urbana (Sim) ou rural (Nao) conforme cadastro PRF',
    'pessoas':               'Total de pessoas envolvidas no acidente',
    'mortos':                'Numero de mortos confirmados',
    'feridos_leves':         'Numero de feridos leves',
    'feridos_graves':        'Numero de feridos graves',
    'feridos':               'Total de feridos (= feridos_leves + feridos_graves, coluna derivada)',
    'ilesos':                'Numero de ilesos',
    'ignorados':             'Pessoas com estado nao informado',
    'veiculos':              'Numero de veiculos envolvidos',
    'latitude':              'Latitude do acidente (decimal negativo)',
    'longitude':             'Longitude do acidente (decimal negativo)',
    'regional':              'Superintendencia regional da PRF',
    'delegacia':             'Delegacia de policia rodoviaria responsavel',
    'uop':                   'Unidade operacional da PRF',
    'mes':                   'Mes do acidente (1=jan, 5=mai) derivado de data_inversa',
    'dia_semana_num':        'Dia da semana numerico (0=segunda, 6=domingo)',
    'fim_de_semana':         '1 se sabado ou domingo, 0 caso contrario',
    'hora':                  'Hora do acidente (0-23) derivada de horario',
    'faixa_horaria':         'Faixa de 6h: Madrugada/Manha/Tarde/Noite (D4)',
    'tracado_via_norm':      'tracado_via com componentes ordenados alfabeticamente',
    'macrocausa':            'Macrocategoria de causa_acidente (D4, aprovada pela equipe)',
    'teve_morte':            '1 se houve pelo menos um morto, 0 caso contrario',
    'grave_ou_fatal':        '1 se ferido_grave > 0 ou morto > 0 (variavel-alvo sugerida para ML)',
}

linhas = [
    '# Dicionario de Dados: datatran_sul_tratado.csv',
    '',
    '| Variavel | Tipo | Origem | Descricao |',
    '|---|---|---|---|',
]
for col in df.columns:
    origem = 'Original' if col in colunas_orig else 'Derivada'
    desc   = descricoes.get(col, '(sem descricao)')
    linhas.append(f'| `{col}` | {df[col].dtype} | {origem} | {desc} |')

with open(DICIONARIO, 'w', encoding='utf-8') as f:
    f.write('\n'.join(linhas))

print(f"Dicionario salvo: {DICIONARIO}")
print(f"Log de decisoes: {LOG_DECISOES}")


---
## 12. Resumo

**Tratamentos aplicados neste notebook:**

| Tratamento | Colunas afetadas | Decisao |
|---|---|---|
| Conversao para datetime + variaveis derivadas | `data_inversa`, `mes`, `dia_semana_num`, `fim_de_semana` | Automatica |
| Extracao de hora e faixa horaria | `hora`, `faixa_horaria` | D4: aprovada |
| Conversao de decimal (virgula para ponto) | `km`, `latitude`, `longitude` | Automatica |
| Normalizacao + dummies de tracado | `tracado_via_norm`, `tracado_*` | Automatica |
| Ausencias disfarcadas -> NaN | `condicao_metereologica`, `sentido_via` | D5: aprovada |
| Recuperacao por municipio | `regional`, `delegacia`, `uop` | Automatica |
| Macrocausas | `macrocausa` | D4: aprovada |
| Variaveis de gravidade | `teve_morte`, `grave_ou_fatal` | D3: aprovada |

**Decisoes ainda em aberto:**
- D1: Periodo (so 2026 ou baixar anos anteriores?)
- D2: Recorte da pergunta (gravidade geral, por estado ou por causa?)

---

> ✍️ **INTERPRETAÇÃO DA EQUIPE:** (a preencher na aula)
> Algum tratamento gerou surpresas? Encontraram alguma inconsistencia inesperada?
> As macrocausas propostas fazem sentido para o contexto de rodovias federais do Sul?
> A escolha de `grave_ou_fatal` como variavel-alvo do modelo esta alinhada com
> o que a PRF precisaria saber na pratica?
